# Medical prognosis based on FL + genetic algorithms

TODO:
- train gen alg on 70% data
- mutation points 

- Convolutional neural networks (?)

(-10,10) -> [0, 1, 2]

-7 -> 0 //
0 -> 1 //
7 -> 2 //

$f(x) = 2\cdot \frac{x + 7}{14}$

In [1]:
import gensim.downloader as api

corpus = api.load('glove-wiki-gigaword-100')

In [5]:
import pandas as pd

data = pd.read_csv('../resources/medical/disease_diagnosis.csv')
data['symptomps_text'] = ""

with open("../resources/medical/symptoms_text_2.txt", "r", encoding="utf-8") as f:
    for line in f:
        index, text = line.strip().split("||")
        index = int(index)
        data.at[index, "symptomps_text"] = text

data.to_csv('../resources/medical/disease_diagnosis_with_symptoms.csv', index=False)
data.head(10)

,Patient_ID,Age,Gender,Symptom_1,Symptom_2,Symptom_3,Heart_Rate_bpm,Body_Temperature_C,Blood_Pressure_mmHg,Oxygen_Saturation_%,Diagnosis,Severity,Treatment_Plan,symptomps_text
0,1,74,Male,Fatigue,Sore throat,Fever,69,39.4,132/91,94,Flu,Moderate,Medication and rest,I've been feeling really off lately. It starte...
1,2,66,Female,Sore throat,Fatigue,Cough,95,39.0,174/98,98,Healthy,Mild,Rest and fluids,I've been feeling really unwell lately. It sta...
2,3,32,Male,Body ache,Sore throat,Fatigue,77,36.8,136/60,96,Healthy,Mild,Rest and fluids,I've been feeling really unwell lately. It sta...
3,4,21,Female,Shortness of breath,Headache,Cough,72,38.9,147/82,99,Healthy,Mild,Rest and fluids,I've been feeling quite unwell lately. It star...
4,5,53,Male,Runny nose,Sore throat,Fatigue,100,36.6,109/106,92,Healthy,Mild,Rest and fluids,I've been feeling really unwell lately. It sta...
5,6,22,Male,Sore throat,Fever,Cough,90,39.5,107/92,93,Flu,Moderate,Medication and rest,"For the past few days, I’ve been feeling prett..."
6,7,21,Male,Sore throat,Fatigue,Cough,71,37.5,126/82,93,Bronchitis,Severe,Hospitalization and medication,I've been feeling really unwell lately. It sta...
7,8,71,Male,Headache,Shortness of breath,Runny nose,64,38.6,153/99,99,Healthy,Mild,Rest and fluids,"For the past few days, I've been feeling quite..."
8,9,56,Female,Shortness of breath,Fever,Headache,103,36.2,152/71,96,Cold,Mild,Rest and fluids,I've been feeling really unwell lately. It sta...
9,10,53,Male,Cough,Fever,Headache,62,39.5,111/104,98,Flu,Moderate,Medication and rest,"For the past few days, I've been feeling reall..."


In [6]:
import random
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import numpy as np
import json

data['Blood_Pressure_mmHg_lower'] = data['Blood_Pressure_mmHg'].apply(lambda x: int(x.split('/')[1]) if isinstance(x, str) and '/' in x else None)
data['Blood_Pressure_mmHg_upper'] = data['Blood_Pressure_mmHg'].apply(lambda x: int(x.split('/')[0]) if isinstance(x, str) and '/' in x else None)
data["Gender_encoded"] = data["Gender"].map({"Male": 0, "Female": 1})

data["Severity_encoded"] = data["Severity"].map({"Mild": 0, "Moderate": 1, "Severe": 2})

feature_list = ["Age", "Heart_Rate_bpm", "Body_Temperature_C", "Blood_Pressure_mmHg_upper", "Blood_Pressure_mmHg_lower", "Oxygen_Saturation_%"]
feature_list_scaled = []

for item in feature_list:
    data[item + "_scaled"] = (data[item] - data[item].min()) / (data[item].max() - data[item].min())
    feature_list_scaled.append(item + "_scaled")

feature_list_scaled.append("Gender_encoded")
feature_list_scaled_indices = {feature_list_scaled[i]: i for i in range(len(feature_list_scaled))}

# data[feature_list_scaled].head()
# print("Feature list (scaled):", feature_list_scaled)

level = ["low", "medium", "high"]
severity = ["mild", "moderate", "severe"]

rate_of_change = np.arange(-10, 10, 1)

antecedents = []
for i in range(len(feature_list_scaled)):
    ant = ctrl.Antecedent(rate_of_change, feature_list_scaled[i])

    ant['low'] = fuzz.trimf(rate_of_change, [-10, -4, 0])
    ant['medium'] = fuzz.trimf(rate_of_change, [-6, 0, 6])
    ant['high'] = fuzz.trimf(rate_of_change, [0, 4, 10])

    ant['neutral'] = fuzz.trimf(rate_of_change, [-10, 0, 10])  # very broad term for fallback

    antecedents.append(ant)

# for item in antecedents:
#     item.automf(names=increase_fallback)

output = ctrl.Consequent(rate_of_change, 'output')
output['mild'] = fuzz.trimf(rate_of_change, [-10, -5, 0])
output['moderate'] = fuzz.trimf(rate_of_change, [-5, 0, 5])
output['severe'] = fuzz.trimf(rate_of_change, [0, 5, 10])

POPULATION_SIZE = 100
RULES_SIZE = 10

def get_random_rule():
    return random.choice(feature_list_scaled) + " " + random.choice(level) + " and " + random.choice(feature_list_scaled) + " " + random.choice(level) + " then " + random.choice(severity)

def get_initial_flrb(rules_size = RULES_SIZE, population_size = POPULATION_SIZE):
    flrb = []
    for i in range(population_size):
        rules = []
        initial_rule = "".join([f + " medium and " for f in feature_list_scaled])
        initial_rule = initial_rule[:-4] + "then moderate"
        rules.append(initial_rule)
        rules += [get_random_rule() for _ in range(rules_size - 1)]
        flrb.append(rules)
    return flrb

def calculate_fuzzy(data_points, rulebase):
    rules = []
    for rule in rulebase:
        predicates = rule.split(' ')
        ant_predicates = antecedents[feature_list_scaled_indices[predicates[0]]][predicates[1]]
        i = 2
        while i < len(predicates) - 2:
            if predicates[i] == 'or':
                ant_predicates = ant_predicates | antecedents[feature_list_scaled_indices[predicates[i+1]]][predicates[i+2]]
            else:
                ant_predicates = ant_predicates & antecedents[feature_list_scaled_indices[predicates[i+1]]][predicates[i+2]]
            i += 3
        rules.append(ctrl.Rule(ant_predicates, output[predicates[len(predicates) - 1]]))

    # Add fallback rule
    rule_default = ctrl.Rule(antecedents[0]['neutral'] | antecedents[1]['neutral'] | antecedents[2]['neutral'] | antecedents[3]['neutral'] | antecedents[4]['neutral'] | antecedents[5]['neutral'] | antecedents[6]['neutral'], output['moderate'])

    # control_system_ctrl = ctrl.ControlSystem([*existing_rules, rule_default])
    # Create the control system and simulation
    ctrl_system = ctrl.ControlSystem([*rules, rule_default])
    ctrl_sim = ctrl.ControlSystemSimulation(ctrl_system)
    output_vector = []

    # Pass inputs to the ControlSystem using Antecedent labels with Pythonic API
    for data_point in data_points:
        for i,point in enumerate(data_point):
            ctrl_sim.input[feature_list_scaled[i]] = point
        try:
            ctrl_sim.compute()
            output_point = ((7 + ctrl_sim.output['output']) / 14) * 2  # Scale to [0, 2]
        except:
            output_point = 0.0
        output_vector.append(output_point)

    # output.view(sim=emotion_sim)
    return output_vector


steps = 20
mutation_steps = [4,5,10,11,20,21,30,31,40,41,50,51,60,61,70,71,80,81,90,91]
flrb = get_initial_flrb()
train_data = data[feature_list_scaled].iloc[:10]

for step in range(steps):
    print(f"Step {step}/{steps}")
    fitness_scores = []
    for rulebase in flrb:
        output_vector = calculate_fuzzy(train_data.values, rulebase)
        mse = np.mean((output_vector - data["Severity_encoded"].values[:len(output_vector)]) ** 2)
        fitness_scores.append(1 / (mse + 1e-6))  # Avoid division by zero

    # Select top 20% of the population
    print(max(fitness_scores))
    top_indices = np.argsort(fitness_scores)[-int(0.2 * POPULATION_SIZE):]
    top_flrb = [flrb[i] for i in top_indices]

    # Create new population through crossover and mutation
    new_flrb = []
    for _ in range(POPULATION_SIZE):
        parent1 = random.choice(top_flrb)
        parent2 = random.choice(top_flrb)
        crossover_point = random.randint(1, RULES_SIZE - 1)
        child_rules = parent1[:crossover_point] + parent2[crossover_point:]

        # Mutation
        mutation_rate = step in mutation_steps and 0.2 or 0.05  
        if random.random() < mutation_rate:  # Mutation probability
            mutation_point = random.randint(0, RULES_SIZE - 1)
            child_rules[mutation_point] = get_random_rule()

        new_flrb.append(child_rules)

    flrb = new_flrb

with open("../resources/sets/outputs.json", "r") as f:
    data = json.load(f)

data["medical"]["minimal-frlb"] = flrb[0]
print(flrb[0])

with open("../resources/sets/outputs.json", "w") as f:
    json.dump(data, f, indent=4)

Step 0/20
2.1227005197676654
Step 1/20
2.123176348428589
Step 2/20
2.1243553930632397
Step 3/20
2.1243553930632397
Step 4/20
2.1243553930632397
Step 5/20
2.124491526913019
Step 6/20
2.124491526913019
Step 7/20
2.124491526913019
Step 8/20
2.124491526913019
Step 9/20
2.124491526913019
Step 10/20
2.124491526913019
Step 11/20
2.125880063340691
Step 12/20
2.125880063340691
Step 13/20
2.125880063340691
Step 14/20
2.125880063340691
Step 15/20
2.125880063340691
Step 16/20
2.125880063340691
Step 17/20
2.125880063340691
Step 18/20
2.125880063340691
Step 19/20
2.125880063340691
['Body_Temperature_C_scaled medium and Oxygen_Saturation_%_scaled medium then mild', 'Heart_Rate_bpm_scaled low and Gender_encoded high then mild', 'Body_Temperature_C_scaled medium and Body_Temperature_C_scaled medium then mild', 'Blood_Pressure_mmHg_upper_scaled medium and Body_Temperature_C_scaled medium then mild', 'Heart_Rate_bpm_scaled low and Oxygen_Saturation_%_scaled medium then mild', 'Gender_encoded high and Bod